In [0]:
CREATE or replace table fk_ce_2026_07_1372.silver.products (
  product_id STRING,
  product_name STRING,
  brand STRING,
  category_l1 STRING,
  category_l2 STRING,
  pack_size INT,
  pack_unit STRING,
  mrp DECIMAL(10,2),
  selling_price DECIMAL(10,2),
  is_active BOOLEAN,
  tags ARRAY<STRING>,
  last_changed_snapshot_date DATE
);
ALTER TABLE fk_ce_2026_07_1372.silver.products SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

MERGE INTO fk_ce_2026_07_1372.silver.products AS t
USING (
  SELECT
    UPPER(TRIM(p.product_id)) AS product_id,
    p.product_name,
    p.brand,
    p.category.l1 AS category_l1,
    p.category.l2 AS category_l2,
    p.pack.size AS pack_size,
    p.pack.unit AS pack_unit,
    p.mrp,
    p.selling_price,
    p.is_active,
    p.tags,
    p.snapshot_date AS last_changed_snapshot_date
  FROM fk_ce_2026_07_1372.bronze.products p
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY UPPER(TRIM(p.product_id))
    ORDER BY p.snapshot_date DESC
  ) = 1
) AS s
ON t.product_id = s.product_id
WHEN MATCHED AND (
  t.product_name <> s.product_name OR
  t.brand <> s.brand OR
  t.category_l1 <> s.category_l1 OR
  t.category_l2 <> s.category_l2 OR
  t.pack_size <> s.pack_size OR
  t.pack_unit <> s.pack_unit OR
  t.mrp <> s.mrp OR
  t.selling_price <> s.selling_price OR
  t.is_active <> s.is_active OR
  t.tags <> s.tags
) THEN UPDATE SET
  product_name = s.product_name,
  brand = s.brand,
  category_l1 = s.category_l1,
  category_l2 = s.category_l2,
  pack_size = s.pack_size,
  pack_unit = s.pack_unit,
  mrp = s.mrp,
  selling_price = s.selling_price,
  is_active = s.is_active,
  tags = s.tags,
  last_changed_snapshot_date = s.last_changed_snapshot_date
WHEN NOT MATCHED THEN INSERT (
  product_id, product_name, brand, category_l1, category_l2, pack_size, pack_unit, mrp,
  selling_price, is_active, tags, last_changed_snapshot_date
) VALUES (
  s.product_id, s.product_name, s.brand, s.category_l1, s.category_l2, s.pack_size, s.pack_unit,
  s.mrp, s.selling_price, s.is_active, s.tags, s.last_changed_snapshot_date
)